# ⚙️ Automated Training Report Generator
### NTUC LearningHub — Mini Project 4
**Skills:** Python Automation · Data Aggregation · HTML Report Generation · Business Reporting  
**Use Case:** Auto-generate a formatted weekly learner performance report for management.
---


## 1️⃣ Imports & Setup

In [ ]:
%matplotlib inline
import random, datetime
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Path("report_output").mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")
print("✅ Setup complete")

## 2️⃣ Generate Sample Learner Data

In [ ]:
COURSES = [
    "Fundamentals of Python Programming",
    "Analyse Business Data Using Python",
    "Advanced Analytics & ML Using Python",
    "Deep Learning Models & AI Using Python",
]

NAMES = [
    "Tan Wei Ming","Sarah Lim","Ravi Kumar","Mary Wong","Ahmad Ali",
    "Linda Chen","Kevin Ng","Priya Nair","James Ho","Siti Rahimah",
    "David Lim","Jessica Tan","Muthu K","Rachel Goh","Bryan Ong",
    "Kavitha S","Nurul Ain","Alex Chua","Mei Ling","Raj Patel",
    "Fiona Lim","Eugene Koh","Aisha Malik","Steven Tan","Cheryl Wong",
    "Mohan Das","Grace Yeo","Nicholas Lim","Ramesh Iyer","Joanne Tan",
]

random.seed(42)
records = []
for i, name in enumerate(NAMES):
    course   = COURSES[i % len(COURSES)]
    quiz     = round(random.uniform(50, 100), 1)
    project  = round(random.uniform(55, 100), 1)
    attend   = random.randint(75, 100)
    final    = round(quiz*0.3 + project*0.5 + attend*0.2, 1)
    grade    = "Distinction" if final>=85 else "Merit" if final>=75 else "Pass" if final>=60 else "Fail"
    records.append({"Name":name,"Course":course,"Quiz":quiz,
                    "Project":project,"Attendance":attend,"Final":final,"Grade":grade})

df = pd.DataFrame(records)
print(f"✅ {len(df)} learner records generated")
df.head(8)

## 3️⃣ Analytics Summary

In [ ]:
print("Overall Stats")
print(f"  Total Learners : {len(df)}")
print(f"  Pass Rate      : {(df['Grade']!='Fail').mean()*100:.0f}%")
print(f"  Avg Final Score: {df['Final'].mean():.1f}")
print(f"  Distinctions   : {(df['Grade']=='Distinction').sum()}")

print("\nBy Course:")
course_stats = df.groupby("Course").agg(
    Headcount=("Name","count"),
    Avg_Final=("Final","mean"),
    Pass_Rate=("Grade", lambda x: (x!="Fail").mean()*100)
).round(1)
display(course_stats)

## 4️⃣ Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Grade distribution
grade_order = ["Distinction","Merit","Pass","Fail"]
grade_counts = df["Grade"].value_counts().reindex(grade_order, fill_value=0)
colors = ["#27ae60","#2980b9","#f39c12","#e74c3c"]
axes[0].bar(grade_counts.index, grade_counts.values, color=colors, edgecolor="white")
axes[0].set_title("Grade Distribution", fontweight="bold")
axes[0].set_ylabel("Count")

# Final score distribution
sns.histplot(df["Final"], bins=12, kde=True, ax=axes[1], color="#4e9af1")
axes[1].axvline(df["Final"].mean(), color="red", linestyle="--", label=f"Mean={df['Final'].mean():.1f}")
axes[1].set_title("Final Score Distribution", fontweight="bold"); axes[1].legend()

# Avg score by course
avg_course = df.groupby("Course")["Final"].mean().sort_values()
axes[2].barh(avg_course.index, avg_course.values, color=sns.color_palette("Set2",len(avg_course)))
axes[2].set_title("Avg Final Score by Course", fontweight="bold")
axes[2].set_xlim(0, 100)
[axes[2].text(v+0.5, i, f"{v:.1f}", va="center", fontsize=9) for i,v in enumerate(avg_course)]

plt.tight_layout(); plt.show()

## 5️⃣ Generate HTML Report

In [ ]:
def build_html_report(df):
    now   = datetime.datetime.now().strftime("%d %B %Y, %H:%M")
    total = len(df); passed = (df["Grade"]!="Fail").sum()
    avg   = df["Final"].mean()
    top3  = df.nlargest(3,"Final")[["Name","Final","Grade"]].values

    rows = ""
    for _, r in df.iterrows():
        gc = {"Distinction":"#27ae60","Merit":"#2980b9","Pass":"#f39c12","Fail":"#e74c3c"}.get(r["Grade"],"#555")
        rows += f"<tr><td>{r['Name']}</td><td>{r['Course']}</td><td>{r['Quiz']}</td><td>{r['Project']}</td><td>{r['Attendance']}%</td><td><b>{r['Final']}</b></td><td style='color:{gc};font-weight:bold'>{r['Grade']}</td></tr>"

    top3_html = "".join(f"<li>🏅 <b>{n}</b> — {s} ({g})</li>" for n,s,g in top3)

    return f"""<!DOCTYPE html><html lang='en'><head><meta charset='UTF-8'>
<title>NTUC LearningHub — Training Report</title>
<style>
body{{font-family:'Segoe UI',sans-serif;margin:40px;color:#2c3e50;background:#f5f6fa}}
h1{{color:#1a252f;border-bottom:3px solid #3498db;padding-bottom:10px}}
h2{{color:#2980b9;margin-top:30px}}
.kpi{{display:inline-block;background:white;border-radius:8px;padding:18px 30px;margin:10px;
      box-shadow:0 2px 8px rgba(0,0,0,.1);text-align:center;min-width:120px}}
.kpi .v{{font-size:2em;font-weight:bold;color:#2980b9}}
.kpi .l{{font-size:.85em;color:#7f8c8d;margin-top:4px}}
table{{border-collapse:collapse;width:100%;background:white;border-radius:8px;
       overflow:hidden;box-shadow:0 2px 8px rgba(0,0,0,.08)}}
th{{background:#2980b9;color:white;padding:12px 14px;text-align:left;font-size:.9em}}
td{{padding:10px 14px;border-bottom:1px solid #ecf0f1;font-size:.88em}}
tr:hover td{{background:#eaf4fb}}
ul{{list-style:none;padding:0}}li{{padding:6px 0}}
footer{{margin-top:40px;font-size:.8em;color:#95a5a6;text-align:center}}
</style></head><body>
<h1>🎓 NTUC LearningHub — Python Training Performance Report</h1>
<p>Generated: <b>{now}</b></p>
<h2>📊 Key Performance Indicators</h2>
<div>
  <div class='kpi'><div class='v'>{total}</div><div class='l'>Total Learners</div></div>
  <div class='kpi'><div class='v'>{passed}</div><div class='l'>Passed</div></div>
  <div class='kpi'><div class='v'>{total-passed}</div><div class='l'>Failed</div></div>
  <div class='kpi'><div class='v'>{avg:.1f}</div><div class='l'>Avg Final Score</div></div>
  <div class='kpi'><div class='v'>{passed/total*100:.0f}%</div><div class='l'>Pass Rate</div></div>
</div>
<h2>🏆 Top 3 Learners</h2><ul>{top3_html}</ul>
<h2>📋 Learner Results</h2>
<table><thead><tr><th>Name</th><th>Course</th><th>Quiz(30%)</th><th>Project(50%)</th>
<th>Attendance(20%)</th><th>Final</th><th>Grade</th></tr></thead>
<tbody>{rows}</tbody></table>
<footer>Auto-generated · NTUC LearningHub Python Trainer Portfolio · [Your Name]</footer>
</body></html>"""

html = build_html_report(df)
out  = Path("report_output/training_report.html")
out.write_text(html, encoding="utf-8")
print(f"✅ Report saved → {out}")
print(f"   Open in any browser to view the formatted management report.")
print(f"   Total: {len(df)} learners | Pass Rate: {(df['Grade']!='Fail').mean()*100:.0f}%")

## 6️⃣ Export Results to CSV & Excel

In [ ]:
# CSV export
df.to_csv("report_output/learner_results.csv", index=False)

# Excel export (multi-sheet)
try:
    with pd.ExcelWriter("report_output/learner_results.xlsx", engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="All Learners", index=False)
        course_stats.to_excel(writer, sheet_name="Course Summary")
    print("✅ Excel report saved → report_output/learner_results.xlsx")
except ImportError:
    print("ℹ  openpyxl not installed — CSV saved successfully")

print("\n📂 Files generated in report_output/:")
for f in sorted(Path("report_output").iterdir()):
    print(f"   {f.name}")